In [4]:
print("Simpal")

Simpal


In [3]:
import pandas as pd

base_path = r"C:\Users\DELL\Desktop\Vendor Performance & Inventory Analytics\data"

begin_inventory = pd.read_csv(f"{base_path}\\begin_inventory.csv")
end_inventory = pd.read_csv(f"{base_path}\\end_inventory.csv")
purchase_prices = pd.read_csv(f"{base_path}\\purchase_prices.csv")
purchases = pd.read_csv(f"{base_path}\\purchases.csv")
sales = pd.read_csv(f"{base_path}\\sales.csv")
vendor_invoice = pd.read_csv(f"{base_path}\\vendor_invoice.csv")

In [6]:
print("NEGATIVE / INVALID NUMERIC VALUES")

print("\nSALES")
print(
    sales[
        (sales["SalesQuantity"] < 0) |
        (sales["SalesDollars"] < 0) |
        (sales["SalesPrice"] < 0) |
        (sales["ExciseTax"] < 0)
    ].shape
)

print("\nPURCHASES")
print(
    purchases[
        (purchases["Quantity"] < 0) |
        (purchases["PurchasePrice"] < 0) |
        (purchases["Dollars"] < 0)
    ].shape
)

print("\nBEGIN INVENTORY")
print(
    begin_inventory[
        (begin_inventory["onHand"] < 0) |
        (begin_inventory["Price"] < 0)
    ].shape
)

print("\nEND INVENTORY")
print(
    end_inventory[
        (end_inventory["onHand"] < 0) |
        (end_inventory["Price"] < 0)
    ].shape
)

print("\nPURCHASE PRICES")
print(
    purchase_prices[
        (purchase_prices["Price"] < 0) |
        (purchase_prices["PurchasePrice"] < 0)
    ].shape
)

print("\nVENDOR INVOICE")
print(
    vendor_invoice[
        (vendor_invoice["Quantity"] < 0) |
        (vendor_invoice["Dollars"] < 0) |
        (vendor_invoice["Freight"] < 0)
    ].shape
)

NEGATIVE / INVALID NUMERIC VALUES

SALES
(0, 14)

PURCHASES
(0, 16)

BEGIN INVENTORY
(0, 9)

END INVENTORY
(0, 9)

PURCHASE PRICES
(0, 9)

VENDOR INVOICE
(0, 10)


In [8]:
print("DATE VALIDATION")

date_columns = {
    "begin_inventory": ["startDate"],
    "end_inventory": ["endDate"],
    "purchases": ["PODate", "ReceivingDate", "InvoiceDate", "PayDate"],
    "sales": ["SalesDate"],
    "vendor_invoice": ["InvoiceDate", "PODate", "PayDate"]
}

datasets = {
    "begin_inventory": begin_inventory,
    "end_inventory": end_inventory,
    "purchases": purchases,
    "sales": sales,
    "vendor_invoice": vendor_invoice
}

for name, cols in date_columns.items():

    print(f"\n--- {name} ---")

    for col in cols:

        original = datasets[name][col]

        converted = pd.to_datetime(original, errors="coerce")

        invalid_count = converted.isna().sum()

        print(
            f"{col}: "
            f"invalid={invalid_count}, "
            f"min={converted.min()}, "
            f"max={converted.max()}"
        )

DATE VALIDATION

--- begin_inventory ---
startDate: invalid=0, min=2024-01-01 00:00:00, max=2024-01-01 00:00:00

--- end_inventory ---
endDate: invalid=0, min=2024-12-31 00:00:00, max=2024-12-31 00:00:00

--- purchases ---
PODate: invalid=0, min=2023-12-20 00:00:00, max=2024-12-23 00:00:00
ReceivingDate: invalid=0, min=2024-01-01 00:00:00, max=2024-12-31 00:00:00
InvoiceDate: invalid=0, min=2024-01-04 00:00:00, max=2025-01-10 00:00:00
PayDate: invalid=0, min=2024-02-04 00:00:00, max=2025-02-19 00:00:00

--- sales ---
SalesDate: invalid=0, min=2024-01-01 00:00:00, max=2024-12-31 00:00:00

--- vendor_invoice ---
InvoiceDate: invalid=0, min=2024-01-04 00:00:00, max=2025-01-10 00:00:00
PODate: invalid=0, min=2023-12-20 00:00:00, max=2024-12-23 00:00:00
PayDate: invalid=0, min=2024-02-04 00:00:00, max=2025-02-19 00:00:00


### Python business consistency checks
Sales calculation check

Sales dollars should broadly agree with:

SalesQuantity × SalesPrice

In [9]:
sales["calculated_sales"] = (
    sales["SalesQuantity"] * sales["SalesPrice"]
)

sales["sales_difference"] = (
    sales["SalesDollars"] - sales["calculated_sales"]
)

print(
    sales["sales_difference"]
    .abs()
    .describe()
)

count    1.282536e+07
mean     5.314854e-16
std      7.783755e-15
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.637979e-12
Name: sales_difference, dtype: float64


In [ ]:
print(
    "Sales records with difference > 0.01:",
    (sales["sales_difference"].abs() > 0.01).sum()
)
# Then check how many are materially different

Sales records with difference > 0.01: 0


In [11]:
# Purchase calculation check

purchases["calculated_dollars"] = (
    purchases["Quantity"] * purchases["PurchasePrice"]
)

purchases["purchase_difference"] = (
    purchases["Dollars"] - purchases["calculated_dollars"]
)

print(
    purchases["purchase_difference"]
    .abs()
    .describe()
)

print(
    "Purchase records with difference > 0.01:",
    (purchases["purchase_difference"].abs() > 0.01).sum()
)

count    2.372474e+06
mean     5.545287e-15
std      2.448177e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.637979e-12
Name: purchase_difference, dtype: float64
Purchase records with difference > 0.01: 0


In [12]:
# Vendor consistency
print(
    purchases.groupby("VendorNumber")["VendorName"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

VendorNumber
2000    2
1587    2
4425    2
60      1
2       1
105     1
388     1
480     1
516     1
653     1
660     1
1003    1
200     1
54      1
1189    1
1128    1
1392    1
1265    1
1439    1
1485    1
Name: VendorName, dtype: int64


In [13]:
print(
    purchase_prices.groupby("VendorNumber")["VendorName"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

VendorNumber
1703    3
1587    2
2000    2
4425    2
60      1
2       1
54      1
480     1
200     1
287     1
105     1
1002    1
1003    1
1128    1
1189    1
1265    1
516     1
653     1
388     1
1439    1
Name: VendorName, dtype: int64


In [14]:
# Brand consistency
print(
    purchase_prices.groupby("Brand")["Description"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

Brand
90631    1
58       1
60       1
61       1
62       1
90013    1
90012    1
90011    1
90010    1
50354    1
47090    1
47075    1
47049    1
47027    1
47014    1
47011    1
47009    1
46985    1
46964    1
46950    1
Name: Description, dtype: int64


In [15]:
# ============================================================
# DATA VALIDATION - OBSERVATIONS
# ============================================================

# 1. EXACT DUPLICATES
# No exact duplicate rows were found in any of the six datasets.
# Therefore, no duplicate rows need to be removed at this stage.


# 2. NEGATIVE / INVALID NUMERIC VALUES
# No negative or invalid numeric values were found in the checked columns
# of Sales, Purchases, Begin Inventory, End Inventory, Purchase Prices,
# or Vendor Invoice.


# 3. DATE VALIDATION
# All checked date columns were successfully converted to datetime.
# No invalid/null dates were detected during date conversion.
#
# Observed date ranges:
# - Begin Inventory: 2024-01-01
# - End Inventory: 2024-12-31
# - Sales: 2024-01-01 to 2024-12-31
# - Purchases: PO dates begin in Dec-2023; receiving/invoice/pay dates
#   extend into 2025.
# - Vendor Invoice shows similar date coverage.
#
# The 2025 invoice/pay dates are not treated as errors at this stage.


# 4. SALES CALCULATION VALIDATION
# SalesDollars was compared with:
# SalesQuantity × SalesPrice
#
# Result:
# 0 records had a difference greater than 0.01.
#
# Observation:
# SalesDollars is consistent with the quantity and sales-price fields
# across the checked sales records.


# 5. PURCHASE CALCULATION VALIDATION
# Dollars was compared with:
# Quantity × PurchasePrice
#
# Result:
# 0 records had a difference greater than 0.01.
#
# Observation:
# Purchase Dollars is consistent with Quantity and PurchasePrice
# across the checked purchase records.


# 6. MISSING VALUES
# Missing values were found in a few columns:
#
# - end_inventory.City              → 1,284 missing
# - purchase_prices.Description     → 1 missing
# - purchase_prices.Size            → 1 missing
# - purchase_prices.Volume          → 1 missing
# - purchases.Size                  → 3 missing
# - vendor_invoice.Approval         → 5,169 missing
#
# Observation:
# These values will NOT be filled or deleted blindly.
# Their business meaning should be investigated before cleaning.


# 7. END INVENTORY CITY
# The missing City values in end_inventory were observed mainly for
# Store 46 in the records inspected.
#
# Observation:
# This suggests the missing values may have a pattern and may be recoverable
# from another dataset/source.


# 8. PURCHASE PRICES MISSING PRODUCT INFORMATION
# One purchase_prices record has missing Description, Size and Volume.
# The record belongs to Brand 4202.
#
# Observation:
# Brand 4202 should be checked in other datasets before deciding whether
# the missing values can be recovered.


# 9. PURCHASES MISSING SIZE
# Only 3 purchase records have missing Size values.
#
# Observation:
# These records should be checked against purchase_prices and inventory
# tables to see whether the correct Size can be recovered.


# 10. VENDOR INVOICE APPROVAL
# 5,169 of 5,543 vendor_invoice rows have missing Approval values.
#
# Observation:
# Approval is missing for the majority of records.
# This should not automatically be treated as bad data.
# We need to understand what the Approval field represents before deciding
# whether NULL should remain NULL or be transformed.


# 11. VENDOR CONSISTENCY
# Some VendorNumber values were associated with more than one VendorName:
#
# Purchases:
# - VendorNumber 2000 → 2 names
# - VendorNumber 1587 → 2 names
# - VendorNumber 4425 → 2 names
#
# Purchase Prices:
# - VendorNumber 1703 → 3 names
# - VendorNumber 1587 → 2 names
# - VendorNumber 2000 → 2 names
# - VendorNumber 4425 → 2 names
#
# Observation:
# This may be caused by differences in vendor-name formatting,
# such as leading/trailing spaces, but it must be investigated before
# calling it a data-quality error.


# 12. BRAND / PRODUCT CONSISTENCY
# The initial displayed brand-description check showed one description
# per Brand for the sampled results.
#
# Observation:
# No clear brand-description conflict was observed in the displayed output.
# A complete check is still required before declaring this fully validated.


# ============================================================
# OVERALL OBSERVATION
# ============================================================
#
# The datasets appear numerically and structurally clean in the checks
# performed so far:
#
# - No exact duplicate rows
# - No negative numeric values in checked fields
# - No invalid dates detected
# - Sales amount calculation is consistent
# - Purchase amount calculation is consistent
#
# The main areas requiring further investigation are:
#
# 1. Missing values
# 2. Vendor-name consistency
# 3. Brand/product relationships
# 4. Table join relationships
#
# No data has been deleted or permanently modified during this validation.
# ============================================================